# 05 Millepora on the 20-genus baseline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CoralsINPA/RedSeaCoralIdentifier/blob/main/notebooks/05_millepora.ipynb)

The Millepora analysis redone on the **correct predictions**: the 20-genus baseline written by `02_zero_shot.ipynb` (`<run>/predictions/finetuned-bioclip__taxonomy_v1__hard20__all.parquet`, correct crops on landscape and portrait photos). It replaces `notebooks/legacy/Millepora_Analysis.ipynb`, which analyzed the output of an excluded earlier pipeline. No GPU is needed. Run the first cell, then the others in order.

**Safe to run and re-run.** It reads the shared Drive and only creates new files: each run writes a new folder `<run>/reports/millepora_<date>/`.

| Step | What it does | Time | Reads | Writes |
|---|---|---|---|---|
| Setup | mounts Drive, updates the clone, installs `coralid[viz]` | 1-3 min | GitHub | nothing on Drive |
| Config | checks the baseline is the `hard20` baseline of the config (stops otherwise) and prints its source | seconds | the baseline and its `.meta.json`, the manifest | nothing |
| Overview | how many crops the baseline calls Millepora, by orientation and confidence band; the runner-ups; a confidence histogram | seconds | the above | the report folder |
| Crop PDFs | Millepora predictions sampled per confidence band, and the likely misses (crops where Millepora is the runner-up, closest first) | 2-5 min | photos, mask JSONs | the report folder |

**What this notebook cannot tell you:** how many of these predictions are right. That needs the blind review sheet of `03_expert_review.ipynb` (on hold, to be combined with the larger expert review). The PDFs here are for looking, not for scoring: they show the model's answer.

**What to paste back:** the printed overview (counts, bands, runner-ups) and the names of the files in the report folder.


In [ ]:
import os
import subprocess
import sys

# ================================ SETTINGS: the only place to edit ================================
DATA_ROOT = "/content/drive/MyDrive/colab notebooks/CoralsMasking"  # the Drive folder CoralsMasking
BRANCH = "main"
REPO_URL = "https://github.com/CoralsINPA/RedSeaCoralIdentifier"
REPO_DIR = "/content/RedSeaCoralIdentifier"
EXTRAS = "viz"
# ==================================================================================================

# The package cannot install itself before it exists. A clone left by an earlier session of this
# runtime is brought to the branch head BEFORE anything is imported from it.
if os.path.isdir(f"{REPO_DIR}/.git"):
    steps = [
        ["git", "-C", REPO_DIR, "fetch", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "checkout", BRANCH],
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "submodule", "update", "--init"],
    ]
else:
    steps = [["git", "clone", "--recurse-submodules", "--branch", BRANCH, REPO_URL, REPO_DIR]]
for step in steps:
    print("$", " ".join(step))
    subprocess.run(step, check=True)
sys.path.insert(0, f"{REPO_DIR}/src")  # enough to import coralid.colab (standard library only)

from coralid.colab import setup  # noqa: E402

env = setup(
    EXTRAS, data_root=DATA_ROOT, branch=BRANCH, repo_url=REPO_URL, repo_dir=REPO_DIR, update=False
)

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from coralid.config import load_config
from coralid.manifest import read_manifest
from coralid.orientation import add_orientation
from coralid.reports import new_report_folder
from coralid.review_source import baseline_source

cfg = load_config(f"{REPO_DIR}/configs/iou60_sta55_pts32.yaml")  # reads CORALID_DATA_ROOT
GENUS = "Millepora"
BANDS = [0.0, 0.5, 0.85, 1.0]  # confidence bands for the overview and the PDFs
PER_BAND = 24  # crops per band in the PDF of predictions
N_MISSES = 48  # crops in the PDF of likely misses
SEED = 0

source = baseline_source(cfg)  # stops unless this is the hard20 baseline of the config
print("Predictions used:")
print("\n".join(source.lines()))
manifest = read_manifest(cfg.manifest_path)
predictions = add_orientation(pd.read_parquet(source.path), manifest)
predictions["uid"] = predictions["image_name"] + "#" + predictions["coral_id"].astype(str)
print(f"{len(predictions)} crops on {predictions['image_name'].nunique()} photos")

## Overview

How many crops the baseline calls Millepora, where, and how confidently; what it would have said otherwise (the runner-up).

In [ ]:
report = new_report_folder(cfg.reports_dir, "millepora")
mil = predictions[predictions["predicted_genus"] == GENUS].copy()
mil["band"] = pd.cut(mil["confidence"], BANDS, include_lowest=True)
print(f"{GENUS}: {len(mil)} of {len(predictions)} crops ({len(mil) / len(predictions):.1%})")
print("\nBy orientation:")
table = pd.DataFrame(
    {"crops": predictions["orientation"].value_counts(), GENUS: mil["orientation"].value_counts()}
)
table = table.fillna(0).astype(int)
table["share"] = (table[GENUS] / table["crops"]).round(3)
print(table.to_string())
print("\nBy confidence band:")
print(mil["band"].value_counts(sort=False).to_string())
print("\nRunner-up genus of the Millepora predictions:")
print(mil["runner_up"].value_counts().head(10).to_string())
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(mil["confidence"], bins=25, color="teal", edgecolor="white")
ax.set(xlabel="confidence", ylabel="crops", title=f"{GENUS} predictions ({len(mil)})")
fig.tight_layout()
fig.savefig(report / "confidence_histogram.png", dpi=120)
plt.show()
mil.drop(columns=["band"]).to_csv(report / "millepora_predictions.csv", index=False)
print(f"\nReport folder: {report}")

## Crop PDFs

A random sample of the Millepora predictions in each confidence band, and the crops where Millepora came second, closest first (possible misses). Captions show the model's answer.

In [ ]:
from coralid.masks import AnnotationStore
from coralid.viz import crop_grid_pdf

store = AnnotationStore(cfg.json_dir)
groups = mil.groupby("band", observed=True)
sample = pd.concat([g.sample(min(len(g), PER_BAND), random_state=SEED) for _, g in groups])
sample = sample.sort_values("confidence")
sample["caption"] = (
    sample["uid"]
    + "  "
    + sample["confidence"].round(2).astype(str)
    + "  2nd: "
    + sample["runner_up"]
)
crop_grid_pdf(
    sample,
    report / "millepora_by_band.pdf",
    store,
    cfg.images_dir,
    cfg.crops,
    title=f"{GENUS} predictions by confidence band",
)
misses = predictions[predictions["runner_up"] == GENUS].nlargest(N_MISSES, GENUS).copy()
misses["caption"] = (
    misses["uid"]
    + "  "
    + misses["predicted_genus"]
    + " "
    + misses["confidence"].round(2).astype(str)
    + "  M: "
    + misses[GENUS].round(2).astype(str)
)
crop_grid_pdf(
    misses,
    report / "likely_misses.pdf",
    store,
    cfg.images_dir,
    cfg.crops,
    title=f"Crops with {GENUS} as runner-up (closest first)",
)
misses.to_csv(report / "likely_misses.csv", index=False)
print(json.dumps({"band_sample": len(sample), "likely_misses": len(misses)}))
print("\n".join(sorted(p.name for p in report.iterdir())))

## Next

- **Precision** (how many Millepora predictions are right) needs blind judgments: the review sheet of `03_expert_review.ipynb`, drawn from the same baseline. It is on hold until it is combined with the larger expert review, so that Assaf reviews once.
- The PDFs above show the model's answer, so a judgment made while looking at them is not blind. Use them to see what the model calls Millepora and what it nearly calls Millepora.